In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os
from PIL import Image
"""from tensorflow import keras
from tensorflow.keras import layers"""
import imagehash
from collections import defaultdict

In [ ]:
class_names = sorted([
    d for d in os.listdir("wikiart")
    if os.path.isdir(os.path.join("wikiart", d))
])

counts = {}
for cls in class_names:
    path = os.path.join("wikiart", cls)
    counts[cls] = len(os.listdir(path))

for cls, n in counts.items():
    print(f"{cls:<30} {n}")

print(f"\nTotal: {sum(counts.values())}")

In [ ]:
for cls in class_names:
    img_path = os.path.join("wikiart", cls, os.listdir(os.path.join("wikiart", cls))[0])
    img = Image.open(img_path)
    print(f"{cls:<30} {img.size} {img.mode}")

In [ ]:
corrupted = []
for cls in class_names:
    cls_path = os.path.join("wikiart", cls)
    for fname in os.listdir(cls_path):
        try:
            Image.open(os.path.join(cls_path, fname)).verify()
        except Exception:
            corrupted.append(os.path.join(cls, fname))

print(f"Corrupted: {len(corrupted)}")
for f in corrupted:
    print(f)

In [ ]:
fig, axes = plt.subplots(len(class_names), 3, figsize=(9, len(class_names) * 3))

for i, cls in enumerate(class_names):
    cls_path = os.path.join("wikiart", cls)
    samples = os.listdir(cls_path)[:3]
    for j, fname in enumerate(samples):
        img = Image.open(os.path.join(cls_path, fname))
        axes[i, j].imshow(img)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_title(cls, fontsize=8, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
sizes = set()
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        w, h = Image.open(os.path.join(cls_path, fname)).size
        sizes.add((w, h))

print(f"Unique sizes found: {sizes}")

In [ ]:
print(f"{'Artist':<30} {'Mean':>8} {'Std':>8}")
print("-" * 48)

for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    means = []
    stds  = []
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)).convert("L"))
        means.append(img.mean())
        stds.append(img.std())
    print(f"{cls:<30} {np.mean(means):>8.1f} {np.mean(stds):>8.1f}")

In [ ]:
print(f"{'Artist':<30} {'R':>8} {'G':>8} {'B':>8} {'Dominant':>10}")
print("-" * 68)

for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    r_means, g_means, b_means = [], [], []
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)))
        r_means.append(img[:,:,0].mean())
        g_means.append(img[:,:,1].mean())
        b_means.append(img[:,:,2].mean())
    
    r = np.mean(r_means)
    g = np.mean(g_means)
    b = np.mean(b_means)
    dominant = ["R", "G", "B"][np.argmax([r, g, b])]
    print(f"{cls:<30} {r:>8.1f} {g:>8.1f} {b:>8.1f} {dominant:>10}")

In [ ]:
print(f"{'Artist':<30} {'R-B gap':>10} {'Warmth':>12}")
print("-" * 55)

for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    r_means, b_means = [], []
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)))
        r_means.append(img[:,:,0].mean())
        b_means.append(img[:,:,2].mean())
    
    gap = np.mean(r_means) - np.mean(b_means)
    warmth = "very warm" if gap > 40 else "warm" if gap > 20 else "neutral"
    print(f"{cls:<30} {gap:>10.1f} {warmth:>12}")

In [ ]:
grayscale_disguised = []
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)))
        if np.allclose(img[:,:,0], img[:,:,1]) and np.allclose(img[:,:,1], img[:,:,2]):
            grayscale_disguised.append(os.path.join(cls, fname))

print(f"Grayscale-as-RGB images: {len(grayscale_disguised)}")

In [ ]:
grayscale_per_artist = defaultdict(int)
for path in grayscale_disguised:
    artist = path.split(os.sep)[0]
    grayscale_per_artist[artist] += 1

print(f"{'Artist':<30} {'Grayscale':>10} {'Total':>8} {'%':>6}")
print("-" * 58)
for cls in class_names:
    n = grayscale_per_artist.get(cls, 0)
    total = counts[cls]
    print(f"{cls:<30} {n:>10} {total:>8} {n/total*100:>5.1f}%")

In [ ]:
hashes = {}  # path → hash

for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        fpath = os.path.join(cls_path, fname)
        try:
            img = Image.open(fpath).convert("RGB")
            h = imagehash.phash(img)  # perceptual hash
            hashes[fpath] = h
        except Exception:
            pass

print(f"Hashes computed: {len(hashes)}")

In [ ]:
THRESHOLD = 8  # 0 = identical, higher = more similar pairs found

duplicates = defaultdict(list)
paths = list(hashes.keys())

for i in range(len(paths)):
    for j in range(i + 1, len(paths)):
        dist = hashes[paths[i]] - hashes[paths[j]]
        if dist <= THRESHOLD:
            duplicates[paths[i]].append((paths[j], dist))

print(f"\nDuplicate groups found: {len(duplicates)}")
for original, matches in list(duplicates.items())[:5]:  # show first 5
    print(f"\n  {original}")
    for match, dist in matches:
        print(f"    → {match}  (distance={dist})")

In [ ]:
by_dist = defaultdict(list)
for original, matches in duplicates.items():
    for match, dist in matches:
        by_dist[dist].append((original, match))

for dist in sorted(by_dist.keys()):
    print(f"Distance {dist}: {len(by_dist[dist])} pairs")

In [ ]:
def show_pairs_by_distance(by_dist, distance, n=None):
    pairs = by_dist[distance]
    if n is not None:
        pairs = pairs[:n]
    
    n_pairs = len(pairs)
    fig, axes = plt.subplots(n_pairs, 2, figsize=(8, n_pairs * 4))
    
    # handle case of only 1 pair
    if n_pairs == 1:
        axes = [axes]
    
    for i, (original, match) in enumerate(pairs):
        img1 = Image.open(original)
        img2 = Image.open(match)
        
        axes[i][0].imshow(img1)
        axes[i][0].set_title(f"{os.path.basename(original)}", fontsize=7)
        axes[i][0].axis("off")
        
        axes[i][1].imshow(img2)
        axes[i][1].set_title(f"{os.path.basename(match)}", fontsize=7)
        axes[i][1].axis("off")
    
    plt.suptitle(f"Duplicate pairs — distance={distance} ({n_pairs} pairs)", 
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
confirmed_pairs = []
for dist in [0, 2, 4]:
    confirmed_pairs.extend(by_dist.get(dist, []))

n = len(confirmed_pairs)
fig, axes = plt.subplots(n, 2, figsize=(8, n * 4))
for i, (original, match) in enumerate(confirmed_pairs):
    axes[i][0].imshow(Image.open(original))
    axes[i][0].set_title(f"{os.path.basename(original)}", fontsize=7)
    axes[i][0].axis("off")
    axes[i][1].imshow(Image.open(match))
    axes[i][1].set_title(f"{os.path.basename(match)}", fontsize=7)
    axes[i][1].axis("off")
plt.suptitle(f"Confirmed duplicates — distance 0-4 ({n} pairs)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
for dist in [6, 8]:
    show_pairs_by_distance(by_dist, distance=dist)

In [ ]:
THRESHOLD_HIGH = 10

duplicates_high = defaultdict(list)
paths = list(hashes.keys())

for i in range(len(paths)):
    for j in range(i + 1, len(paths)):
        dist = hashes[paths[i]] - hashes[paths[j]]
        if dist <= THRESHOLD_HIGH:
            duplicates_high[paths[i]].append((paths[j], dist))

# Show only NEW pairs not found at threshold=8
by_dist_high = defaultdict(list)
for original, matches in duplicates_high.items():
    for match, dist in matches:
        by_dist_high[dist].append((original, match))

print(f"Total pairs at threshold=10 : {sum(len(v) for v in by_dist_high.values())}")
print(f"Total pairs at threshold=8  : {sum(len(v) for v in by_dist.values())}")
print(f"New pairs at distance 9-10  : {len(by_dist_high.get(9, [])) + len(by_dist_high.get(10, []))}")

# Visualise only the new ones
for dist in [9, 10]:
    if by_dist_high.get(dist):
        show_pairs_by_distance(by_dist_high, distance=dist)

In [ ]:
# All false positives (both directions) — filenames only
false_positives = {
    # from distance 6-8
    ("camille-pissarro_portrait-of-nini-1884.jpg", "ilya-repin_portrait-of-chuguev-resident-s-l-lyubitskaya-1877.jpg"),
    ("rembrandt_man-with-a-magnifying-glass-1629.jpg", "rembrandt_portrait-of-rembrandt-s-father.jpg"),
    ("ilya-repin_portrait-of-vasily-kirillovich-syutayev-1882.jpg", "john-singer-sargent_sir-george-lewis-1896.jpg"),
    ("rembrandt_self-portrait-4.jpg", "rembrandt_self-portrait-with-hat-and-gold-chain-1633.jpg"),
    ("albrecht-durer_holy-family-2.jpg", "ilya-repin_prophet-1890.jpg"),
    ("ilya-repin_portrait-of-painter-grigory-grigoryevich-myasoyedov-1883.jpg", "ilya-repin_portrait-of-vasily-kirillovich-syutayev-1882.jpg"),
    ("ilya-repin_portrait-of-painter-grigory-grigoryevich-myasoyedov-1883.jpg", "john-singer-sargent_sir-george-lewis-1896.jpg"),
    ("john-singer-sargent_mary-turner-austin-1880.jpg", "rembrandt_portrait-of-a-man-with-a-golden-helmet-1648.jpg"),
    ("rembrandt_self-portrait-1.jpg", "rembrandt_portrait-of-a-young-woman.jpg"),
    # from distance 9-10
    ("albrecht-durer_cervus-lucanus.jpg", "vincent-van-gogh_peasant-woman-seen-against-the-window-two-heads-1885.jpg"),
    ("boris-kustodiev_portrait-of-the-painter-ivan-bilibin-1901.jpg", "nicholas-roerich_zmievna-1906.jpg"),
    ("camille-pissarro_portrait-of-nini-1884.jpg", "edgar-degas_portrait-of-rene-de-gas-1855.jpg"),
    ("camille-pissarro_portrait-of-nini-1884.jpg", "rembrandt_self-portrait-5.jpg"),
    ("camille-pissarro_the-hay-wagon-montfoucault-1879.jpg", "pierre-auguste-renoir_seated-nude-1913.jpg"),
    ("camille-pissarro_laundresses-at-eragny.jpg", "salvador-dali_equestrian-fantasy-portrait-of-lady-dunn.jpg"),
    ("camille-pissarro_peasant-working-in-the-fields.jpg", "eugene-boudin_dordrecht-the-great-church-from-the-canal.jpg"),
    ("childe-hassam_the-isles-of-shoals.jpg", "claude-monet_cliff-near-pourville.jpg"),
    ("childe-hassam_field-of-poppies-isles-of-shaos-appledore.jpg", "nicholas-roerich_himalayas-ladakh.jpg"),
    ("claude-monet_self-portrait-with-a-beret-1886.jpg", "john-singer-sargent_villa-torlonia-fountain-1907.jpg"),
    ("edgar-degas_dancer-with-a-fan-1.jpg", "marc-chagall_untitled.jpg"),
    ("edgar-degas_portrait-of-rene-de-gas-1855.jpg", "rembrandt_self-portrait-1660-2.jpg"),
    ("edgar-degas_sky-study.jpg", "ilya-repin_raising-of-jairus-daughter-1871.jpg"),
    ("edgar-degas_seated-young-man-in-a-jacket-with-an-umbrella.jpg", "pablo-picasso_guitar-1920.jpg"),
    ("edgar-degas_study-for-semiramis-building-babylon-1861.jpg", "salvador-dali_self-potrait.jpg"),
    ("eugene-boudin_petit-port-de-saint-jean-near-villefranche-1892.jpg", "marc-chagall_lid-meeting-of-isaac-and-rebecca-1980.jpg"),
    ("eugene-boudin_seascape-1871.jpg", "nicholas-roerich_tibet-evening-1937jpg"),
    ("eugene-boudin_seascape-1871.jpg", "nicholas-roerich_tibet-1937-2.jpg"),
    ("gustave-dore_all-dwellings-else-flood-overwhelmed-and-them-with-all-their-pomp-deep-under-water-rolled.jpg", "ilya-repin_warrior-xvii-century-1879.jpg"),
    ("gustave-dore_pluto-and-virgil(1).jpg", "pierre-auguste-renoir_landscape-18.jpg"),
    ("ilya-repin_portrait-of-chuguev-resident-s-l-lyubitskaya-1877.jpg", "pablo-picasso_portrait-of-aunt-pepa-1896.jpg"),
    ("ilya-repin_portrait-of-vasily-kirillovich-syutayev-1882.jpg", "vincent-van-gogh_self-portrait-with-pipe-and-glass-1887.jpg"),
    ("ilya-repin_portrait-of-a-bocharova-artist-s-aunts-1859.jpg", "rembrandt_portrait-of-a-young-woman-1655.jpg"),
    ("ilya-repin_barge-haulers-by-campfire-1870.jpg", "vincent-van-gogh_peasant-with-a-chopping-knife-1881.jpg"),
    ("john-singer-sargent_sir-philip-sasson-phillip-albert-gustave-david-sasson-1923.jpg", "rembrandt_portrait-of-hendrickje-stoffels-1660.jpg"),
    ("john-singer-sargent_sir-philip-sasson-phillip-albert-gustave-david-sasson-1923.jpg", "rembrandt_portrait-of-a-young-woman-1655.jpg"),
    ("john-singer-sargent_sir-george-lewis-1896.jpg", "paul-cezanne_portrait-of-a-man-1864-1.jpg"),
    ("john-singer-sargent_mary-turner-austin-1880.jpg", "rembrandt_bust-of-an-old-man-1631.jpg"),
    ("john-singer-sargent_mary-tumner-austin-1880.jpg", "rembrandt_portrait-of-rembrandt-s-father.jpg"),
    ("john-singer-sargent_portrait-of-lady-sassoon-1907.jpg", "john-singer-sargent_helen-brice-1907.jpg"),
    ("john-singer-sargent_hugh-lane-1906.jpg", "rembrandt_portrait-of-rembrandt-s-father.jpg"),
    ("martiros-saryan_view-from-noramberda-1959.jpg", "nicholas-roerich_sared-himalayas-1933.jpg"),
    ("martiros-saryan_gathering-of-grapes-1935.jpg", "vincent-van-gogh_corinthian-capital-1863(1).jpg"),
    ("nicholas-roerich_kangchenjunga-1937-1.jpg", "vincent-van-gogh_wheat-field-with-cypresses-1889.jpg"),
    ("pablo-picasso_two-brothers-1906.jpg", "pierre-auguste-renoir_madame-paul-berard.jpg"),
    ("pablo-picasso_portrait-of-aunt-pepa-1896.jpg", "rembrandt_self-portrait-1660-2.jpg"),
    ("pablo-picasso_woman-with-cigarette-1903.jpg", "vincent-van-gogh_view-of-paris-from-vincent-s-room-in-the-rue-lepic-1887jpg"),
    ("raphael-kirchner_clovers-1899-3.jpg", "vincent-van-gogh_cypresses-with-four-people-working-in-the-field-1890(1).jpg"),
    ("raphael-kirchner_girls-heads-in-a-circle-1901-4.jpg", "raphael-kirchner_marcelle-earle-1.jpg"),
    ("raphael-kirchner_princess-riquette.jpg", "vincent-van-gogh_cottages-1883(1).jpg"),
    ("rembrandt_hendrickje-stoffels-in-velvet-beret.jpg", "rembrandt_portrait-of-hendrikje-stoffels-1654.jpg"),
    ("rembrandt_a-woman-seated-before-a-dutch-stove-1658.jpg", "rembrandt_portrait-of-lieven-willemsz-van-coppenol-1653.jpg"),
    ("rembrandt_self-portrait-1.jpg", "rembrandt_self-portrait-1633.jpg"),
    ("rembrandt_self-portrait-1633.jpg", "rembrandt_self-portrait-with-hat-and-gold-chain-1633.jpg"),
    ("rembrandt_portrait-of-hendrickje-stoffels-1660.jpg", "rembrandt_portrait-of-a-young-woman-1655.jpg"),
    ("rembrandt_the-persian-1632.jpg", "vincent-van-gogh_woman-with-hat-half-length-1886.jpg"),
    ("rembrandt_portrait-of-rembrandt-s-father.jpg", "rembrandt_self-portrait-1669.jpg"),
}

# Add reverse directions
false_positives_both = set()
for p1, p2 in false_positives:
    false_positives_both.add((p1, p2))
    false_positives_both.add((p2, p1))

# Build removal list
to_remove = set()
for original, matches in duplicates_high.items():
    for match, dist in matches:
        orig_fname  = os.path.basename(original)
        match_fname = os.path.basename(match)
        if (orig_fname, match_fname) not in false_positives_both:
            # keep the more colorful one
            img_orig  = np.array(Image.open(original))
            img_match = np.array(Image.open(match))
            colorfulness_orig  = np.std([img_orig[:,:,0].mean(),  img_orig[:,:,1].mean(),  img_orig[:,:,2].mean()])
            colorfulness_match = np.std([img_match[:,:,0].mean(), img_match[:,:,1].mean(), img_match[:,:,2].mean()])
            if colorfulness_orig >= colorfulness_match:
                to_remove.add(match)
            else:
                to_remove.add(original)

print(f"Files to remove: {len(to_remove)}")

# Preview before deleting
for f in list(to_remove)[:10]:
    print(f)

In [ ]:
for fpath in to_remove:
    if os.path.exists(fpath):
        os.remove(fpath)
        print(f"Removed: {fpath}")

print(f"\nTotal removed: {len(to_remove)}")

In [ ]:
hashes_clean = {}
for root, dirs, files in os.walk('wikiart'):
    for fname in files:
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            fpath = os.path.join(root, fname)
            try:
                hashes_clean[fpath] = imagehash.phash(Image.open(fpath))
            except Exception as e:
                print(f"Could not hash {fpath}: {e}")

print(f"Total images after cleanup: {len(hashes_clean)}")

In [ ]:
THRESHOLD_HIGH = 10
duplicates_high = defaultdict(list)
paths = list(hashes_clean.keys())

for i in range(len(paths)):
    for j in range(i + 1, len(paths)):
        orig_fname  = os.path.basename(paths[i])
        match_fname = os.path.basename(paths[j])
        if (orig_fname, match_fname) in false_positives_both:
            continue
        dist = hashes_clean[paths[i]] - hashes_clean[paths[j]]
        if dist <= THRESHOLD_HIGH:
            duplicates_high[paths[i]].append((paths[j], dist))

In [ ]:
by_dist_high = defaultdict(list)
for original, matches in duplicates_high.items():
    for match, dist in matches:
        by_dist_high[dist].append((original, match))

total = sum(len(v) for v in by_dist_high.values())
print(f"Total pairs found: {total}")

In [ ]:
dark_bright = []
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)).convert("L"))
        mean = img.mean()
        if mean < 20 or mean > 240:
            dark_bright.append((os.path.join(cls, fname), round(mean, 1)))

print(f"Extremely dark/bright images: {len(dark_bright)}")
for path, mean in dark_bright[:10]:
    print(f"  {path}  mean={mean}")

In [ ]:
cols = 5
rows = -(-len(dark_bright) // cols)  # ceiling division

fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
axes = axes.flatten()

for i, (path, mean) in enumerate(dark_bright):
    img = mpimg.imread(os.path.join('wikiart', path))
    axes[i].imshow(img)
    axes[i].set_title(f"mean={mean}", fontsize=8)
    axes[i].axis("off")

# Hide any unused subplots
for i in range(len(dark_bright), len(axes)):
    axes[i].axis("off")

plt.suptitle(f"Extremely dark/bright images ({len(dark_bright)} total)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
low_info = []
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)).convert("L"))
        if img.std() < 10:
            low_info.append((os.path.join(cls, fname), round(img.std(), 2)))

print(f"Low information images: {len(low_info)}")
for path, std in low_info[:10]:
    print(f"  {path}  std={std}")

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))

for ax, (path, std) in zip(axes.flatten(), low_info):
    img = Image.open(os.path.join('wikiart', path))
    ax.imshow(img)
    ax.set_title(f"{path.split(os.sep)[1]}\nstd={std}", fontsize=6)
    ax.axis("off")

# hide unused axes
for ax in axes.flatten()[len(low_info):]:
    ax.axis("off")

plt.suptitle("Low information images", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
"""artist_means = {}
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    imgs = [np.array(Image.open(os.path.join(cls_path, f)).convert("L")).mean() 
            for f in os.listdir(cls_path)[:50]]
    artist_means[cls] = np.mean(imgs)

outlier_images = []
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)).convert("L"))
        diff = abs(img.mean() - artist_means[cls])
        if diff > 80:
            outlier_images.append((os.path.join(cls, fname), round(diff, 1)))

print(f"Outlier images: {len(outlier_images)}")"""

In [ ]:
artist_means = {}
artist_stds = {}
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    imgs = [np.array(Image.open(os.path.join(cls_path, f)).convert("L")).mean() 
            for f in os.listdir(cls_path)]  # ← all images
    artist_means[cls] = np.mean(imgs)
    artist_stds[cls] = np.std(imgs)

# Use z-score instead of absolute threshold
outlier_images = []
for cls in class_names:
    cls_path = os.path.join('wikiart', cls)
    for fname in os.listdir(cls_path):
        img = np.array(Image.open(os.path.join(cls_path, fname)).convert("L"))
        z_score = abs(img.mean() - artist_means[cls]) / (artist_stds[cls] + 1e-6)
        if z_score > 3:  # standard 3-sigma rule
            outlier_images.append((os.path.join(cls, fname), round(z_score, 2)))

print(f"Outlier images: {len(outlier_images)}")

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(15, 12))

for ax, (path, diff) in zip(axes.flatten(), outlier_images[:20]):
    full_path = os.path.join('wikiart', path)
    img = Image.open(full_path)
    artist = path.split(os.sep)[0]
    ax.imshow(img)
    ax.set_title(f"{artist}\ndiff={diff}", fontsize=6)
    ax.axis("off")

for ax in axes.flatten()[20:]:
    ax.axis("off")

plt.suptitle("Outlier images (brightness far from artist mean)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

outlier_artists = Counter([path.split(os.sep)[0] for path, _ in outlier_images])
for artist, n in outlier_artists.most_common():
    print(f"{artist:<30} {n}")

In [ ]:
import shutil

shutil.copytree('wikiart', 'wikiart_clean')